In [33]:
def relative_l2_error(model, exact_fn, n_x=200, n_t=100,
                      domain_x=(0, np.pi), domain_t=(0, 1)):
    x = np.linspace(domain_x[0], domain_x[1], n_x)
    t = np.linspace(domain_t[0], domain_t[1], n_t)
    X, T = np.meshgrid(x, t)
    inputs = np.stack([X.ravel(), T.ravel()], axis=1).astype(np.float32) # расчетная сетка из 20000 точек (200x100)
    p_pinn  = model(inputs).numpy().reshape(X.shape)
    p_exact = exact_fn(X, T)
    return np.linalg.norm(p_pinn - p_exact) / np.linalg.norm(p_exact)

In [34]:
import pandas as pd
import numpy as np
x = np.linspace(0, np.pi, 200)
t = np.linspace(0, 1, 100)
X, T = np.meshgrid(x, t)
# print(pd.DataFrame(X[:5, :5]))
# print(pd.DataFrame(T[:5, :5]))

df = pd.DataFrame({
    'x': X.ravel(),
    't': T.ravel()
})

# a = [3, 4]
# np.linalg.norm(a)
# inputs = pd.DataFrame(np.stack([X.ravel(), T.ravel()], axis=1).astype(np.float32))
# inputs
# df.head()

## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \sin(x)$, boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [35]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Dirichlet)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  inputs_left = tf.concat([x_left, t], axis=1)
  inputs_right = tf.concat([x_right, t], axis=1)
  p_left = model(inputs_left)
  p_right = model(inputs_right)

  return tf.reduce_mean(
      tf.square(p_left)
  ) + tf.reduce_mean(
      tf.square(p_right)
  )

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.sin(x)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history_dirichlet = []
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss_dirichlet = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss_dirichlet, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    loss_val = float(loss_dirichlet.numpy())
    loss_history_dirichlet.append(loss_val)
    print(f"Epoch {epoch}, Loss: {loss_dirichlet.numpy()}")

err_dirichlet = relative_l2_error(model, lambda x, t: np.exp(-t) * np.sin(x))
print(f"Dirichlet: relative L2 error = {err_dirichlet:.4e}")

Epoch 0, Loss: 3.0231518745422363
Epoch 50, Loss: 0.3311985731124878
Epoch 100, Loss: 0.25508493185043335
Epoch 150, Loss: 0.23652410507202148
Epoch 200, Loss: 0.1923695206642151
Epoch 250, Loss: 0.1423485279083252
Epoch 300, Loss: 0.1352294385433197
Epoch 350, Loss: 0.09474076330661774
Epoch 400, Loss: 0.08978321403265
Epoch 450, Loss: 0.07433106750249863
Epoch 500, Loss: 0.0718589723110199
Epoch 550, Loss: 0.06456024199724197
Epoch 600, Loss: 0.05883726477622986
Epoch 650, Loss: 0.04117182642221451
Epoch 700, Loss: 0.037617333233356476
Epoch 750, Loss: 0.040020883083343506
Epoch 800, Loss: 0.032183028757572174
Epoch 850, Loss: 0.03182594105601311
Epoch 900, Loss: 0.023136399686336517
Epoch 950, Loss: 0.02254847064614296
Dirichlet: relative L2 error = 7.3582e-02


In [36]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.sin(x_plot)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## 1.2. Piezoconductivity Equation (Neumann)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \cos(x)$, boundary conditions $p_x(0,t) = 0$, $p_x(\pi,t) = 0$.

In [37]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Neumann)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x_left)
    tape.watch(x_right)
    inputs_left = tf.concat([x_left, t], axis=1)
    inputs_right = tf.concat([x_right, t], axis=1)
    p_left = model(inputs_left)
    p_right = model(inputs_right)

    p_x_left = tape.gradient(p_left, x_left)
    p_x_right = tape.gradient(p_right, x_right)
  del tape

  return tf.reduce_mean(
      tf.square(p_x_left)
  ) + tf.reduce_mean(
      tf.square(p_x_right)
  )

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.cos(x)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history_neumann = []
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss_neumann = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss_neumann, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    loss_val = float(loss_neumann.numpy())
    loss_history_neumann.append(loss_val)
    print(f"Epoch {epoch}, Loss: {loss_neumann.numpy()}")

err_neumann = relative_l2_error(model, lambda x, t: np.exp(-t) * np.cos(x))
print(f"Neumann: relative L2 error = {err_neumann:.4e}")

Epoch 0, Loss: 0.6869577169418335
Epoch 50, Loss: 0.2747601270675659
Epoch 100, Loss: 0.18625691533088684
Epoch 150, Loss: 0.133529931306839
Epoch 200, Loss: 0.10291940718889236
Epoch 250, Loss: 0.0824931338429451
Epoch 300, Loss: 0.06906647980213165
Epoch 350, Loss: 0.054812170565128326
Epoch 400, Loss: 0.045591700822114944
Epoch 450, Loss: 0.034401532262563705
Epoch 500, Loss: 0.029074721038341522
Epoch 550, Loss: 0.02059137634932995
Epoch 600, Loss: 0.017188748344779015
Epoch 650, Loss: 0.01481950655579567
Epoch 700, Loss: 0.00991133227944374
Epoch 750, Loss: 0.007618824951350689
Epoch 800, Loss: 0.008500928990542889
Epoch 850, Loss: 0.005870341788977385
Epoch 900, Loss: 0.00374821899458766
Epoch 950, Loss: 0.003785576205700636
Neumann: relative L2 error = 3.3719e-02


In [38]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.cos(x_plot)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## 1.3. Piezoconductivity Equation (Robin)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \cos\left(x - \frac{\pi}{4}\right)$, boundary conditions $p_x(0,t) - p(0,t) = 0$, $p_x(\pi,t) - p(\pi,t) = 0$.

In [39]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Robin)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x_left)
    tape.watch(x_right)
    inputs_left = tf.concat([x_left, t], axis=1)
    inputs_right = tf.concat([x_right, t], axis=1)
    p_left = model(inputs_left)
    p_right = model(inputs_right)

    p_x_left = tape.gradient(p_left, x_left)
    p_x_right = tape.gradient(p_right, x_right)
  del tape

  loss_left = tf.reduce_mean(tf.square(p_x_left - p_left))
  loss_right = tf.reduce_mean(tf.square(p_x_right - p_right))

  return loss_left + loss_right

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.cos(x - np.pi / 4)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
loss_history_robin = []
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss_robin = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss_robin, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    loss_val = float(loss_robin.numpy())
    loss_history_robin.append(loss_val)
    print(f"Epoch {epoch}, Loss: {loss_robin.numpy()}")

err_robin = relative_l2_error(model, lambda x, t: np.exp(-t) * np.cos(x - np.pi / 4))
print(f"Robin: relative L2 error = {err_robin:.4e}")

Epoch 0, Loss: 2.72860050201416
Epoch 50, Loss: 0.3675285875797272
Epoch 100, Loss: 0.29798272252082825
Epoch 150, Loss: 0.26936712861061096
Epoch 200, Loss: 0.17046724259853363
Epoch 250, Loss: 0.15330737829208374
Epoch 300, Loss: 0.1326708197593689
Epoch 350, Loss: 0.11048495024442673
Epoch 400, Loss: 0.0838993713259697
Epoch 450, Loss: 0.05818511173129082
Epoch 500, Loss: 0.043150126934051514
Epoch 550, Loss: 0.029148664325475693
Epoch 600, Loss: 0.02341661974787712
Epoch 650, Loss: 0.018333595246076584
Epoch 700, Loss: 0.015224754810333252
Epoch 750, Loss: 0.01756300777196884
Epoch 800, Loss: 0.015379497781395912
Epoch 850, Loss: 0.012828371487557888
Epoch 900, Loss: 0.01019381359219551
Epoch 950, Loss: 0.008373505435883999
Robin: relative L2 error = 1.4740e-01


In [40]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.cos(x_plot - np.pi / 4)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## LOSS:

In [41]:
import plotly.graph_objects as go

epochs = [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500,
          550, 600, 650, 700, 750, 800, 850, 900, 950]

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=loss_history_dirichlet, mode='lines+markers', name='Дирихле'))
fig.add_trace(go.Scatter(x=epochs, y=loss_history_neumann, mode='lines+markers', name='Нейман'))
fig.add_trace(go.Scatter(x=epochs, y=loss_history_robin, mode='lines+markers', name='Робин'))
fig.update_layout(
    title='Сходимость обучения PINN',
    xaxis_title='Эпоха',
    yaxis_title='Функция потерь',
    yaxis_type='log',  # логарифмическая шкала
    template='plotly_white'
)
fig.show()